# Process the goggles-disparity session

Builds the `DisparityDevice` (disparity-conditioned), runs
`DisparitySessionAnalyzer.analyze_session_stats`, and saves
`{session_name}_dataframe.pkl` next to the session data. That df is **long**
(one row per cell x disparity); `CrossExperimentAnalysis` collapses it to
reliable-in-any-disparity. It is what
`monocular-disparity-distance-comparison.ipynb` reads for this experiment via
`CrossExperimentAnalysis.from_devices(R, M, D)`.

Run this in its own kernel with **only** `disparity-analysis` on `sys.path`.

In [1]:
import sys, os
import pandas as pd

base_dir = r"C:\Users\acostamendoza\projects\distance-project"
for p in (base_dir, base_dir + r"\disparity-analysis"):
    if p not in sys.path:
        sys.path.append(p)

from src.data.DisparityDevice import DisparityDevice
from src.analysis.DisparityAnalysis import DisparitySessionAnalyzer

%load_ext autoreload
%autoreload 2

In [2]:
mouse, date, experiment = "SAM045", "260721", "goggles-disparity"
D = DisparityDevice(mouse_name=mouse, record_date=date, experiment_name=experiment)
print("session:", D.session_name, "| disparities:", D.disparities, "| cells:", len(D.suite2p_id))

Directory results already exists
session: 260721_SAM045_goggles-disparity | disparities: [-5, 0, 5, 10, 15] | cells: 1305


In [3]:
# load cached stats if present, else compute once (slow) and save the pkl the
# comparison notebook reads
pkl_path = os.path.join(D.data_dir, f"{D.session_name}_dataframe.pkl")
if os.path.exists(pkl_path):
    stats_df = pd.read_pickle(pkl_path)
    print(f"Loaded cached stats: {pkl_path}")
else:
    stats_df = DisparitySessionAnalyzer(D).analyze_session_stats(n_iterations=200)
    DisparitySessionAnalyzer(D).save_dataframe(stats_df)   # -> pkl_path
stats_df.head()

Saved DataFrame to: D:/data-distance-coding-project/processed-data\260721_SAM045_goggles-disparity\260721_SAM045_goggles-disparity_dataframe.pkl


,cell,session_name,disparity,reliable,mean_corr,rf,rf_value,rf_deg
0,0,260721_SAM045_goggles-disparity,-5,False,0.657045,48,0.880565,43.220339
1,0,260721_SAM045_goggles-disparity,0,False,0.206819,52,0.466816,49.322034
2,0,260721_SAM045_goggles-disparity,5,False,0.031304,59,0.318114,60.000000
3,0,260721_SAM045_goggles-disparity,10,False,-0.273635,0,0.246288,-30.000000
4,0,260721_SAM045_goggles-disparity,15,False,0.493714,52,0.214938,49.322034
